# **XGBoost vs LightBoost vs CatBoost**


In [2]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.1 MB/s eta 0:00:00


In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier


In [4]:

# ============================================================
# 1. LOAD MNIST
# ============================================================
print("=" * 60)
print("Loading MNIST Dataset...")
print("=" * 60)

X, y = fetch_openml(
    "mnist_784",
    version=1,
    return_X_y=True,
    as_frame=False,
    parser="auto"
)

X = X.astype(np.float32)
y = y.astype(np.int32)

# Normalize pixels
X /= 255.0

print(f"Dataset Shape: {X.shape}")
print(f"Labels Shape : {y.shape}")


Loading MNIST Dataset...
Dataset Shape: (70000, 784)
Labels Shape : (70000,)


In [5]:
# ============================================================
# 2. TRAIN TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"\nTraining Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")


Training Samples : 56000
Testing Samples  : 14000


In [6]:
# ============================================================
# 3. STANDARD SCALING
# ============================================================
print("\nApplying StandardScaler...")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Applying StandardScaler...


In [7]:
# ============================================================
# 4. PCA
# ============================================================
print("Applying PCA (99% Variance)...")

pca = PCA(
    n_components=0.99,
    svd_solver="full",
    random_state=42
)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original Features : {X_train.shape[1]}")
print(f"PCA Features      : {X_train_pca.shape[1]}")
print(
    f"Explained Variance: "
    f"{pca.explained_variance_ratio_.sum():.4f}"
)

Applying PCA (99% Variance)...
Original Features : 784
PCA Features      : 542
Explained Variance: 0.9901


In [8]:
# ============================================================
# 5. XGBOOST GPU
# ============================================================
print("\n" + "=" * 60)
print("Training XGBoost (GPU)")
print("=" * 60)

xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=10,

    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method="hist",
    device="cuda",

    eval_metric="mlogloss",
    early_stopping_rounds=20,

    random_state=42
)

xgb_model.fit(
    X_train_pca,
    y_train,
    eval_set=[(X_test_pca, y_test)],
    verbose=False
)

xgb_preds = xgb_model.predict(X_test_pca)

xgb_acc = accuracy_score(y_test, xgb_preds)

print(f"XGBoost Accuracy: {xgb_acc:.4f}")

# ============================================================
# 6. LIGHTGBM GPU
# ============================================================
print("\n" + "=" * 60)
print("Training LightGBM (GPU)")
print("=" * 60)

try:

    lgb_model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=10,

        n_estimators=500,
        learning_rate=0.05,

        num_leaves=127,
        max_depth=-1,

        device="gpu",

        random_state=42,
        verbose=-1
    )

    lgb_model.fit(
        X_train_pca,
        y_train
    )

    lgb_preds = lgb_model.predict(X_test_pca)

    lgb_acc = accuracy_score(y_test, lgb_preds)

    print(f"LightGBM Accuracy: {lgb_acc:.4f}")

except Exception as e:

    print("\nGPU LightGBM not available.")
    print("Error:", e)

    print("\nFalling back to CPU...")

    lgb_model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=10,

        n_estimators=500,
        learning_rate=0.05,

        num_leaves=127,
        max_depth=-1,

        random_state=42,
        verbose=-1
    )

    lgb_model.fit(
        X_train_pca,
        y_train
    )

    lgb_preds = lgb_model.predict(X_test_pca)

    lgb_acc = accuracy_score(y_test, lgb_preds)

    print(f"LightGBM Accuracy: {lgb_acc:.4f}")

# ============================================================
# 7. CATBOOST GPU
# ============================================================
print("\n" + "=" * 60)
print("Training CatBoost (GPU)")
print("=" * 60)

cb_model = CatBoostClassifier(
    loss_function="MultiClass",

    iterations=500,
    learning_rate=0.05,
    depth=8,

    task_type="GPU",
    devices="0",

    random_seed=42,
    verbose=100
)

cb_model.fit(
    X_train_pca,
    y_train
)

cb_preds = cb_model.predict(X_test_pca)

cb_preds = cb_preds.astype(np.int32).flatten()

cb_acc = accuracy_score(y_test, cb_preds)

print(f"CatBoost Accuracy: {cb_acc:.4f}")



Training XGBoost (GPU)
XGBoost Accuracy: 0.9596

Training LightGBM (GPU)
LightGBM Accuracy: 0.9624

Training CatBoost (GPU)
0:	learn: 2.0953258	total: 251ms	remaining: 2m 5s
100:	learn: 0.3272827	total: 13s	remaining: 51.3s
200:	learn: 0.1899565	total: 26s	remaining: 38.7s
300:	learn: 0.1444155	total: 37.7s	remaining: 25s
400:	learn: 0.1184610	total: 48.8s	remaining: 12s
499:	learn: 0.1020360	total: 59.4s	remaining: 0us
CatBoost Accuracy: 0.9503


In [9]:

# ============================================================
# 8. RESULTS COMPARISON
# ============================================================
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

results = pd.DataFrame({
    "Model": [
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "Accuracy": [
        xgb_acc,
        lgb_acc,
        cb_acc
    ]
})

results = results.sort_values(
    by="Accuracy",
    ascending=False
)

print(results)



MODEL COMPARISON
      Model  Accuracy
1  LightGBM  0.962357
0   XGBoost  0.959643
2  CatBoost  0.950286


In [10]:

# ============================================================
# 9. BEST MODEL
# ============================================================
best_model = results.iloc[0]["Model"]
best_acc = results.iloc[0]["Accuracy"]

print("\nBest Model:", best_model)
print(f"Best Accuracy: {best_acc:.4f}")



Best Model: LightGBM
Best Accuracy: 0.9624


In [11]:

# ============================================================
# 10. CLASSIFICATION REPORTS
# ============================================================
print("\n" + "=" * 60)
print("XGBOOST CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        xgb_preds
    )
)

print("\n" + "=" * 60)
print("LIGHTGBM CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        lgb_preds
    )
)

print("\n" + "=" * 60)
print("CATBOOST CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        cb_preds
    )
)


XGBOOST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1381
           1       0.99      0.98      0.98      1575
           2       0.95      0.95      0.95      1398
           3       0.94      0.95      0.95      1428
           4       0.96      0.96      0.96      1365
           5       0.96      0.94      0.95      1263
           6       0.97      0.98      0.97      1375
           7       0.96      0.96      0.96      1459
           8       0.94      0.95      0.94      1365
           9       0.94      0.94      0.94      1391

    accuracy                           0.96     14000
   macro avg       0.96      0.96      0.96     14000
weighted avg       0.96      0.96      0.96     14000


LIGHTGBM CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1381
           1       0.99      0.98      0.98      1575
           2   